# Module 01: Khí quyển chuẩn ISA / The International Standard Atmosphere
**AE2.008 · USTH: Aerodynamics and Flight Mechanics**

🇻🇳 Notebook cài đặt ISA bằng code, đối chiếu các con số trong **EASA Module 08** (Hình 1-3, 1-4, 1-7), giải lại các bài **Anderson 1.1, 1.11, 1.12** và dùng `assert` để tự kiểm các đáp án quiz của module.

🇬🇧 This notebook implements ISA in code, checks the numbers in **EASA Module 08** (Figures 1-3, 1-4, 1-7), re-solves **Anderson 1.1, 1.11, 1.12** and uses `assert` to self-check the module's quiz answers.

In [1]:
import math
G0, R_AIR, GAMMA = 9.80665, 287.05287, 1.4
P0, T0, RHO0 = 101325.0, 288.15, 1.225
FT, PSI = 0.3048, 6894.757

def isa(h_m):
    """ISA 0-20 km: (T [K], p [Pa], rho [kg/m3]). Troposphere lapse -6.5 K/km, isothermal 216.65 K above 11 km."""
    if h_m <= 11000:
        L = -0.0065; T = T0 + L*h_m; p = P0*(T/T0)**(-G0/(L*R_AIR))
    else:
        T = 216.65; p11 = P0*(T/T0)**(-G0/(-0.0065*R_AIR)); p = p11*math.exp(-G0*(h_m-11000)/(R_AIR*T))
    return T, p, p/(R_AIR*T)

def isa_ft(h_ft): return isa(h_ft*FT)


## 1. Đơn vị áp suất / Pressure units (EASA Fig. 1-3)

In [2]:
atm = P0
print(f"1 atm = {atm/PSI:.2f} psi = {atm/3386.389:.2f} inHg = {atm/100:.1f} hPa = {atm/133.322:.0f} mmHg")
assert abs(atm/PSI - 14.7) < 0.01 and abs(atm/3386.389 - 29.92) < 0.01 and abs(atm/133.322 - 760) < 0.5

1 atm = 14.70 psi = 29.92 inHg = 1013.2 hPa = 760 mmHg


## 2. Áp suất ở 40 000 ft / Pressure at 40 000 ft (EASA Fig. 1-4: 2.72 psi)

In [3]:
T, p, rho = isa_ft(40000)
print(f"40 000 ft: T = {T-273.15:.1f} C, p = {p/PSI:.3f} psi = {p/P0*100:.1f}% of sea level")
assert abs(p/PSI - 2.72) < 0.01

40 000 ft: T = -56.5 C, p = 2.720 psi = 18.5% of sea level


## 3. Bảng ISA / ISA table (EASA Fig. 1-7)

In [4]:
print(f"{'h [ft]':>8} {'T [C]':>8} {'p [hPa]':>9} {'p [psi]':>8} {'rho [kg/m3]':>12}")
for h in (0, 5000, 10000, 20000, 30000, 36089, 40000):
    T, p, rho = isa_ft(h)
    print(f"{h:>8} {T-273.15:>8.1f} {p/100:>9.1f} {p/PSI:>8.2f} {rho:>12.4f}")
# EASA Fig. 1-7: sea level 14.67 psi, 1013.53 hPa, 1.23 kg/m3; 5000 ft 12.05 psi, 5.1 C
T, p, rho = isa_ft(5000); assert abs(T-273.15-5.1) < 0.2
T, p, rho = isa(11000); assert abs(T-273.15+56.5) < 1e-6 and abs(rho/RHO0 - 0.2971) < 0.001

  h [ft]    T [C]   p [hPa]  p [psi]  rho [kg/m3]
       0     15.0    1013.2    14.70       1.2250
    5000      5.1     843.1    12.23       1.0555
   10000     -4.8     696.8    10.11       0.9046
   20000    -24.6     465.6     6.75       0.6527
   30000    -44.4     300.9     4.36       0.4583
   36089    -56.5     226.3     3.28       0.3639
   40000    -56.5     187.5     2.72       0.3016


## 4. Anderson 1.1 / 1.11 / 1.12

In [5]:
# 1.1a
rho_a = 1.9e4/(287*203); print("1.1a rho =", round(rho_a, 3), "kg/m3"); assert abs(rho_a-0.326) < 5e-4
# 1.1b
T_b = 1058/(1.23e-3*1716); print("1.1b T =", round(T_b), "R"); assert round(T_b) == 501
# 1.11  U-tube: high column on model side -> lower pressure
p_model = 1.01e5 - 1.36e4*9.8*0.2; print("1.11 p =", round(p_model), "N/m2"); assert abs(p_model-7.43e4) < 50
# 1.12  Zeppelin: W = buoyancy + aerodynamic lift
rho1000 = isa(1000)[2]; B = 15000*rho1000*9.8
S = math.pi*14**2/4; q = 0.5*rho1000*30**2; Lift = q*S*0.05
print(f"1.12 rho(1000 m) = {rho1000:.4f}, B = {B:.0f} N, q = {q:.1f}, S = {S:.1f}, L = {Lift:.0f} N, W = {B+Lift:.0f} N")
assert abs(rho1000-1.1117) < 5e-4 and abs(Lift-3847) < 5 and abs((B+Lift)-1.67e5) < 1e3
# The Solution Manual (4th ed.) prints L = 3487 N on one line but uses 3847 N in the next: typo, final answer correct.

1.1a rho = 0.326 kg/m3
1.1b T = 501 R
1.11 p = 74344 N/m2
1.12 rho(1000 m) = 1.1116, B = 163411 N, q = 500.2, S = 153.9, L = 3850 N, W = 167262 N


## 5. Độ cao mật độ / Density altitude (ví dụ minh họa / illustrative example)

In [6]:
T5, p5, _ = isa_ft(5000)
rho_hot = p5/(R_AIR*(30+273.15))
lo, hi = 0.0, 15000.0
for _ in range(80):
    mid = (lo+hi)/2
    if isa(mid)[2] > rho_hot: lo = mid
    else: hi = mid
print(f"5000 ft, 30 C: rho = {rho_hot:.3f} kg/m3 = {rho_hot/RHO0*100:.0f}% of sea level, density altitude = {lo/FT:.0f} ft")
assert 7000 < lo/FT < 9000

5000 ft, 30 C: rho = 0.969 kg/m3 = 79% of sea level, density altitude = 7801 ft


## 6. Kiểm quiz / Quiz self-check

In [7]:
assert abs(isa(1000)[2]-1.1117) < 5e-4            # Anderson 1.12
assert abs(isa(3000)[2]-0.9093) < 5e-4            # used in module 02 (Anderson 4.11)
assert round((P0/100),2) == 1013.25               # 1 atm in hPa
assert abs(isa(11000)[0]-273.15 - (-56.5)) < 1e-6  # ISA tropopause temperature
assert abs((isa_ft(36089)[2]/RHO0) - 0.30) < 0.005 # density ratio at tropopause
print("All Module 01 checks passed.")

All Module 01 checks passed.


## 7. Câu bổ sung / Extra questions

In [8]:
assert abs(isa_ft(10000)[2]-0.9046) < 5e-4                  # density at 10 000 ft
assert abs(isa_ft(20000)[1]/100-465.6) < 0.3                    # hPa at 20 000 ft (EASA 465.4)
assert abs(isa_ft(10000)[0]-273.15+4.8) < 0.05                  # C at 10 000 ft (EASA -4.8)
rho_dev = 101325/(R_AIR*(15+T0)); lo, hi = 0.0, 15000.0
for _ in range(80):
    mid = (lo+hi)/2
    if isa(mid)[2] > rho_dev: lo = mid
    else: hi = mid
print("density altitude at sea level, ISA+15:", round(lo/FT), "ft"); assert 1500 < lo/FT < 3500
print("Extra Module 01 checks passed.")

density altitude at sea level, ISA+15: 1724 ft
Extra Module 01 checks passed.
